# [2] Naive Trial with Qwen-3 just using Prompt

## Imports

In [ ]:
import env

In [ ]:
from epidec.models.qwen3 import ChatHistory, Qwen3Model
from epidec.datasets import SWUnivDaconDataset

from torch.utils.data import DataLoader

import pandas as pd
from tqdm.auto import tqdm

import json
import sys

## Load Datasets

In [ ]:
DATA_ROOT = "./data"

train_dataset = SWUnivDaconDataset(DATA_ROOT, train=True, valid_ratio=0.1)
valid_dataset = SWUnivDaconDataset(DATA_ROOT, valid=True, valid_ratio=0.1)
test_dataset = SWUnivDaconDataset(DATA_ROOT, train=False)

print(f"INFO: Dataset loaded successfully. Train - {len(train_dataset)}, Valid - {len(valid_dataset)}, Test - {len(test_dataset)}")

In [ ]:
train_dataset[1]

In [ ]:
valid_dataset[1]

## Define Model

In [ ]:
query = lambda p: f"""Analyze the following Korean text paragraph to determine if it was written by a human or generated by AI:

**Text:**: {p}"""

In [ ]:
system_prompt = """You are an expert in distinguishing between human-written and AI-generated text. You specialize in detecting AI-generated content by analyzing **domain-specific linguistic pattern leakage** - where AI models inappropriately use vocabulary, expressions, and grammatical patterns that are characteristic of specific domains in contexts where they don't belong.

## Core Detection Principle
AI language models learn domain-specific linguistic patterns (specialized vocabulary, expressions, grammatical structures) but often fail to restrict their usage to appropriate contexts. They may use medical research terminology in casual writing, academic hedging language in news articles, or technical jargon in literary contexts - creating subtle but detectable cross-contamination.

**CRITICAL DISTINCTION:**
- **HUMAN CHARACTERISTIC**: Natural style/tone changes, emotional shifts, contextual formality adjustments
- **AI CHARACTERISTIC**: Inappropriate cross-domain vocabulary and grammatical pattern usage

## Analysis Framework

### 1. Domain-Specific Pattern Leakage Detection
- **Vocabulary Contamination**: Identify words/phrases that are strongly associated with specific domains appearing in inappropriate contexts (e.g., "delve" from medical literature appearing in casual blog-style writing)
- **Grammatical Pattern Mixing**: Detect domain-specific sentence structures being used outside their natural context
- **Expression Transplantation**: Spot formulaic expressions from one domain appearing inappropriately in another

### 2. Domain Pattern Recognition
- **Medical/Scientific**: "delve into", "elucidate", "furthermore", passive constructions, hedging language
- **Academic**: "it is noteworthy that", "considerable attention", nominalizations, complex subordination
- **Legal**: "pursuant to", "heretofore", "aforementioned", formal conditional structures
- **Technical**: "implement", "utilize", procedural language, step-by-step markers
- **Journalistic**: Attribution patterns, inverted pyramid, factual declaratives

### 3. Natural vs Artificial Pattern Usage
- **Natural Human Usage**: Domain patterns appear in contextually appropriate situations
- **Artificial AI Usage**: Domain patterns appear regardless of context appropriateness
- **IMPORTANT**: Style changes, formality shifts, and tone variations are HUMAN characteristics and should NOT be flagged as AI indicators

## Analysis Process

### Step 1: Domain Pattern Inventory
Identify specific vocabulary, expressions, and grammatical patterns that belong to particular domains within the text.

### Step 2: Context Appropriateness Assessment
Evaluate whether each identified domain pattern appears in an appropriate context or represents cross-domain contamination.

### Step 3: Contamination vs Natural Variation
Distinguish between inappropriate domain pattern leakage (AI indicator) and natural human style variation (human indicator).

## Output Format
You must respond with a valid JSON object in the following format:

```json
{
  "domain_patterns": "Identified domain-specific vocabulary, expressions, and grammatical patterns",
  "context_appropriateness": "Assessment of whether domain patterns appear in appropriate contexts",
  "contamination_evidence": "Specific examples of inappropriate cross-domain pattern usage",
  "detection_rationale": "Key evidence for domain pattern contamination vs natural human variation",
  "probability": 0.75
}
```

**Critical Requirements:**
- Always output valid JSON format
- Probability must be a number between 0.0 and 1.0
- Focus on domain pattern contamination, NOT style changes
- Remember: Style/tone changes are human characteristics
- All text fields should be concise but informative
- Do not include any text outside the JSON object

## Important Considerations
- **Style Changes are HUMAN**: Natural formality shifts, tone changes, and contextual adjustments indicate human authorship
- **Domain Pattern Contamination is AI**: Inappropriate usage of domain-specific linguistic patterns indicates AI generation
- **Korean Language Specificity**: Consider Korean-specific domain patterns and expressions
- **Context Matters**: The same expression can be appropriate in one context and inappropriate in another
- **Focus on Subtlety**: Look for subtle vocabulary and grammatical pattern misuse rather than obvious errors
- **Avoid Over-Detection**: Formal, structured writing and repetitive patterns can be natural in certain contexts (Wikipedia, academic writing, etc.)"""

In [ ]:
class ChatHistory(ChatHistory):
    def create_prompt(self, system_prompt: str, user_prompt: str = ""):
        return [dict(role="system", content=system_prompt), *self, dict(role="user", content=query(user_prompt))]

In [ ]:
class Qwen3ModelForTextClassification(Qwen3Model):
    context_length = 4096

    def classify(
        self,
        user_prompt: str,
        grammar: str | None = None,
        temperature: float = 0.6,
        top_p: float = 0.95,
        top_k: int = 20,
        min_p: float = 0,
        typical_p: float = 1.0,
        repeat_penalty: float = 1.0
    ) -> str:
        return "".join(self.chat(
            chat_history=ChatHistory(),
            user_prompt=user_prompt,
            system_prompt="/nothink " + system_prompt,
            tools=[],
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            min_p=min_p,
            typical_p=typical_p,
            stream=True,
            max_new_tokens=0,
            repeat_penalty=repeat_penalty,
            print_output=True,
            grammar=grammar
        ))

    @staticmethod
    def extract_json(response_text: str) -> int:
        try:
            return json.loads(response_text.split("</think>")[-1].strip().replace("```json", "").replace("```", ""))
        except Exception:
            return {}

    def validate(self, dataset: list, retry_count: int = 1, shuffle: bool = False, check_only_for: int | None = None):
        corrects, errors, true_human, false_human, results = [], [], [], [], []
        progress = tqdm(DataLoader(dataset, batch_size=1, shuffle=shuffle), desc="Validating...")

        for idx, data in enumerate(progress):
            label = data[1]
            data = data[0]

            if check_only_for is not None and label != check_only_for:
                continue  # Skip if the label does not match the specified check

            for trial in range(retry_count):
                assistant_reply = self.classify(data)
                predicted = self.extract_json(assistant_reply)
                if predicted: break
                print(f"WARNING: Invalid prediction for index {idx}, retrying... ({trial + 1}/{retry_count})")

            result = dict(question=data, label=label, predicted=predicted)
            predicted_label = 1 if predicted['probability'] >= 0.5 else 0
            if predicted_label == label:
                corrects.append(result)
                print(f"INFO: Correct prediction for index {idx}\n\n")
                if label == 0:
                    true_human.append(result)
            else:
                result = dict(**result, traceback=assistant_reply)
                errors.append(result)
                print(f"ERROR: Incorrect prediction for index {idx}\n\n")
                if label == 0:
                    false_human.append(result)
            results.append(result)
            progress.set_description(f"Correct: {len(corrects)}/{len(results)} [H: {len(true_human)}, A: {len(corrects)-len(true_human)}], Errors: {len(errors)}/{len(results)} [H: {len(false_human)}, A: {len(errors)-len(false_human)}]")

        print(f"INFO: Correct: {len(corrects)}/{len(results)}, Errors: {len(errors)}/{len(results)}")
        return corrects, errors, results

    def test(self, dataset: list | str, retry_count: int = 100):
        results = []
        if isinstance(dataset, str):
            dataset = [dict(question=dataset)]  # Wrap single string input in dict format

        for idx, data in enumerate(tqdm(dataset, desc="Testing...")):
            data = data[0]

            for trial in range(retry_count):
                assistant_reply = self.classify(data)
                predicted = self.extract_json(assistant_reply)
                if predicted: break
                print(f"WARNING: Invalid prediction for index {idx}, retrying... ({trial + 1}/{retry_count})")

            result = dict(question=data, label=predicted['probability'])
            results.append(result)
        return results

In [ ]:
model = Qwen3ModelForTextClassification()

## Evaluation

In [ ]:
# Validation
corrects, errors, results = model.validate(dataset=valid_dataset, shuffle=True, retry_count=3, check_only_for=1)
pd.DataFrame(results)

In [ ]:
# Test
results = model.test(dataset=test_dataset)
pd.DataFrame(results)